# Bipartite Graph Creation

- Creates a bipartite graph NetworkX object (G_bipartite), saves it to /data/
- Creates a member_nodes.csv, saves it to /data/

#### Run Notebooks

In [1]:
%%capture
# Run data wrangling nb
%run data_grace.ipynb

In [ ]:
# %% [markdown]
# #### Corrected Bipartite Generation

# %%
# 1. Clean the Edge List
rsvps_df = rsvps_df.copy()
member_event = rsvps_df.drop(columns=['group_id']).dropna(subset=['member_id', 'event_id'])

# Create namespaced strings: Use the RAW event_id, NOT the integer map
member_event['m_id'] = 'm_' + member_event['member_id'].astype(str).str.replace(r'\.0$', '', regex=True)
member_event['e_id'] = 'e_' + member_event['event_id'].astype(str).str.replace(r'\.0$', '', regex=True)

# 2. Namespace the Metadata Dictionaries
# IMPORTANT: Ensure events_df is indexed by 'event_id' before dictionary creation
events_df_indexed = events_df.copy()
if 'event_id' in events_df_indexed.columns:
    events_df_indexed.set_index('event_id', inplace=True)

mem_dict_clean = {
    'm_' + str(k).replace('.0', ''): v 
    for k, v in mem_df_new.to_dict(orient='index').items()
}
event_dict_clean = {
    'e_' + str(k).replace('.0', ''): v 
    for k, v in events_df_indexed.to_dict(orient='index').items()
}

# 3. Referential Integrity Filter
valid_m_set = set(mem_dict_clean.keys())
valid_e_set = set(event_dict_clean.keys())

member_event_clean = member_event[
    (member_event['m_id'].isin(valid_m_set)) & 
    (member_event['e_id'].isin(valid_e_set))
].copy()

print(f"Pruned {len(member_event) - len(member_event_clean)} edges.")
print(f"Remaining Edges: {len(member_event_clean)}")

# 4. Create Graph
G_bipartite = nx.Graph()
member_nodes = []
event_nodes = []

for m_id, attrs in mem_dict_clean.items():
    G_bipartite.add_node(m_id, type='member', **attrs)
    member_nodes.append(m_id)

for e_id, attrs in event_dict_clean.items():
    G_bipartite.add_node(e_id, type='event', **attrs)
    event_nodes.append(e_id)

G_bipartite.add_edges_from(zip(member_event_clean['m_id'], member_event_clean['e_id']))

# Final check
print(f"Nodes: {G_bipartite.number_of_nodes()} | Edges: {G_bipartite.number_of_edges()}")

import pandas as pd
import networkx as nx
from pathlib import Path
def serialize_bipartite_artifacts(G_bipartite, member_nodes, output_dir="data"):
    """
    Serializes (runs checks, prints to file) the bipartite graph and member nodes.
    Ensures directory existence and enforces strict string typing for node IDs.
    """
    out_path = Path.cwd() / output_dir
    out_path.mkdir(parents=True, exist_ok=True)

    # 1. Cast node attributes
    for _, data in G_bipartite.nodes(data=True):
        for key, value in data.items():
            if isinstance(value, pd.Timestamp):
                data[key] = value.isoformat()

    # 2. Cast edge attributes
    for _, _, data in G_bipartite.edges(data=True):
        for key, value in data.items():
            if isinstance(value, pd.Timestamp):
                data[key] = value.isoformat()

    # 3. Cast graph-level attributes
    for key, value in G_bipartite.graph.items():
        if isinstance(value, pd.Timestamp):
            G_bipartite.graph[key] = value.isoformat()

    # 4. Write member nodes with explicit string casting
    with open(out_path / "member_nodes.csv", 'w') as f:
        for member in member_nodes:
            f.write(f"{str(member)}\n")
            
    # 5. Serialize Graph
    nx.write_graphml(G_bipartite, out_path / "G_bipartite.graphml")

# Serialize
serialize_bipartite_artifacts(G_bipartite, member_nodes)

Pruned 527 edges.
Remaining Edges: 126286
Nodes: 43760 | Edges: 126286


#### Imports

In [3]:

# # Specific to network purposes
# import networkx as nx
# from networkx.algorithms import bipartite
# import torch

#### Data Arrangement

In [4]:
# # Create df from rsvp data but drop group_id column
# rsvps_df=rsvps_df.copy()
# member_event=rsvps_df.drop(columns=['group_id'])


In [5]:
# # Find all the unique event ids in the events df
# event_nodes = events_df.event_id.unique()

# # Create a dictionary to map event ids to integers
# event_nodes_dict = {v: i for i, v in enumerate(event_nodes)}

# # Apply the map to the member_event df and drop the original event id column
# member_event['event_id_int'] = member_event['event_id'].map(event_nodes_dict)
# member_event=member_event.drop(columns=['event_id'])

# # Apply same map to the events df and set the new event id column as the index
# events_df['event_id_int'] = events_df['event_id'].map(event_nodes_dict)
# events_df.set_index('event_id_int', inplace=True,drop=True)

In [6]:
# # Create a dictionary of dictionaries in order to eventually assign as node attributes for members
#     # Pull from one-hot encoded members df
# mem_dict = mem_df_new.to_dict(orient='index')

# # Create a dictionary of dictionaries in order to eventually assign as node attributes for events
# event_dict= events_df.to_dict(orient='index')

#### Bipartite Graph Creation

In [7]:
# # Define empty graph
# G_bipartite = nx.Graph()

# # Create arrays to hold nodes
# member_nodes =[]
# event_nodes = []

# # Add member nodes
# for member_id, attrs in mem_dict.items():
#     G_bipartite.add_node(member_id, type='member', **attrs)
#     member_nodes.append(member_id)

# # Add event nodes 
# for event_id, attrs in event_dict.items():
#     G_bipartite.add_node(event_id, type='event', **attrs)
#     event_nodes.append(event_id)

# # Add edges
# for _,row in member_event.iterrows():
#     G_bipartite.add_edge(row['member_id'], row['event_id_int'])

In [8]:
# A_bipartite = bipartite.biadjacency_matrix(G_bipartite, row_order=member_nodes, column_order=event_nodes)
# A_bipartite_tensor = torch.tensor(A_bipartite.toarray(), dtype=torch.float)

In [9]:
# A_bipartite_tensor

In [10]:
def inspect_node_features(G_bipartite, expected_types=2):
    """
    Prints the attributes of one node for each distinct type present in the graph.
    
    Args:
        G_bipartite (nx.Graph): The loaded bipartite graph.
        expected_types (int): The number of distinct node types to find before 
                              terminating the search. Defaults to 2 for bipartite.
    """
    seen_types = set()
    
    for node_id, attrs in G_bipartite.nodes(data=True):
        # Extract type, falling back to 'unspecified' if missing
        node_type = attrs.get('type', 'unspecified')
        
        if node_type not in seen_types:
            seen_types.add(node_type)
            
            print(f"### Node Type: {node_type.upper()}")
            print(f"ID: {node_id}")
            print("Attributes:")
            
            if not attrs:
                print("  (No attributes found)")
            else:
                for key, value in attrs.items():
                    # Print the key, value, and the underlying data type
                    print(f"  - {key}: {value} | type: {type(value).__name__}")
            
            print("-" * 40)
            
        # Early exit optimization
        if len(seen_types) >= expected_types:
            break

# Execution execution
inspect_node_features(G_bipartite)

### Node Type: MEMBER
ID: m_2069
Attributes:
  - type: member | type: str
  - lat: 36.0 | type: float
  - lon: -86.79 | type: float
  - hometown_alexandria: 0.0 | type: float
  - hometown_annapolis: 0.0 | type: float
  - hometown_antioch: 0.0 | type: float
  - hometown_arlington: 0.0 | type: float
  - hometown_ashland city: 0.0 | type: float
  - hometown_atlanta: 0.0 | type: float
  - hometown_austin: 0.0 | type: float
  - hometown_baltimore: 0.0 | type: float
  - hometown_birmingham: 0.0 | type: float
  - hometown_boston: 0.0 | type: float
  - hometown_bowling green: 0.0 | type: float
  - hometown_brentwood: 1.0 | type: float
  - hometown_brooklyn: 0.0 | type: float
  - hometown_charlotte: 0.0 | type: float
  - hometown_chattanooga: 0.0 | type: float
  - hometown_chicago: 0.0 | type: float
  - hometown_cincinnati: 0.0 | type: float
  - hometown_clarksville: 0.0 | type: float
  - hometown_cleveland: 0.0 | type: float
  - hometown_columbia: 0.0 | type: float
  - hometown_columbus: 0.0 |